# ConvNeXt V1 — A ConvNet for the 2020s

**Paper**: Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., & Xie, S. (2022).  
**"A ConvNet for the 2020s"**  
CVPR 2022 · arXiv:2201.03545  
Meta AI Research

---

## Why This Paper Matters

By 2021, Vision Transformers (ViT, Swin) were outperforming CNN architectures on most benchmarks. The narrative was: **"attention is all you need"** — transformers win because of their global attention mechanism.

ConvNeXt V1 challenged this narrative by asking a different question: **what happens if we give a ResNet all the same design choices as Swin Transformer?**

The answer: a pure CNN matches Swin at every scale — same accuracy, comparable FLOPs, no attention required.  
The improvements came entirely from **training procedure** and **architectural micro-choices**, not from the attention mechanism itself.

ConvNeXt-L achieves **84.3% top-1 on ImageNet** (vs Swin-L's 84.2%).  
ConvNeXt-XL reaches **87.0%** with ImageNet-22K pretraining.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('dark_background')
ACCENT = '#7EB8F7'
RED    = '#F77E7E'
GREEN  = '#7EF7A4'
GOLD   = '#F7C97E'
PURPLE = '#C87EF7'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ConvNeXt V1: Modernising ResNet Step by Step', fontsize=14, color='white', y=1.02)

# --- Panel 1: The modernisation journey ---
ax = axes[0]
ax.set_xlim(0, 6)
ax.set_ylim(0, 15)
ax.axis('off')
ax.set_title('ResNet → ConvNeXt: 7 Steps', color=ACCENT)

# Each step: (accuracy gain, label, color)
journey = [
    ('ResNet-50 baseline', 76.1, '#555'),
    ('+Training recipe\n(AdamW, mixup, RandAug)', 78.8, GOLD),
    ('+Patchify stem\n(4×4 s4 conv)', 79.4, ACCENT),
    ('+Stage ratio [3,3,9,3]\n(Swin-style)', 79.5, ACCENT),
    ('+Inverted bottleneck\n(C→4C→C)', 80.5, GREEN),
    ('+Depthwise 7×7 conv', 80.6, GREEN),
    ('+LayerNorm + fewer acts', 81.3, PURPLE),
    ('ConvNeXt-T (final)\n+separate downsampling', 82.1, GOLD),
]

y_positions = np.linspace(13.5, 0.8, len(journey))
for (label, acc, col), yp in zip(journey, y_positions):
    ax.text(3, yp, f'{label}  →  {acc:.1f}%', ha='center', va='center', fontsize=8.5,
            color='black' if col not in ['#555'] else 'white',
            bbox=dict(boxstyle='round,pad=0.35', facecolor=col, alpha=0.9))

for yp0, yp1 in zip(y_positions[:-1], y_positions[1:]):
    ax.annotate('', xy=(3, yp1+0.4), xytext=(3, yp0-0.4),
                arrowprops=dict(arrowstyle='->', color='#888'))

# --- Panel 2: V1 vs V2 block comparison ---
ax = axes[1]
ax.set_xlim(0, 8)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('V1 Block vs V2 Block', color=ACCENT)

# V1 block
v1_steps = [
    (2, 11, 'DWConv 7×7', ACCENT),
    (2, 9.5, 'LayerNorm', GOLD),
    (2, 8, 'Linear(C→4C)', GREEN),
    (2, 6.5, 'GELU', GREEN),
    (2, 5, 'Linear(4C→C)', GREEN),
    (2, 3.5, '[Layer Scale γ]', PURPLE),
    (2, 2, '+ residual', '#888'),
]
# V2 block
v2_steps = [
    (6, 11, 'DWConv 7×7', ACCENT),
    (6, 9.5, 'LayerNorm', GOLD),
    (6, 8, 'Linear(C→4C)', GREEN),
    (6, 6.5, 'GELU', GREEN),
    (6, 5, 'GRN  ← V2 addition', RED),
    (6, 3.5, 'Linear(4C→C)', GREEN),
    (6, 2, '+ residual', '#888'),
]

ax.text(2, 11.9, 'ConvNeXt V1', ha='center', color=ACCENT, fontsize=11, fontweight='bold')
ax.text(6, 11.9, 'ConvNeXt V2', ha='center', color=RED, fontsize=11, fontweight='bold')

for steps in [v1_steps, v2_steps]:
    for x, y, label, col in steps:
        ax.text(x, y, label, ha='center', va='center', fontsize=8.5,
                color='black' if col not in ['#888'] else 'white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.85))
    ys = [s[1] for s in steps]
    for y0, y1 in zip(ys[:-1], ys[1:]):
        ax.annotate('', xy=(steps[0][0], y1+0.4), xytext=(steps[0][0], y0-0.4),
                    arrowprops=dict(arrowstyle='->', color='#666'))

# Dividing line
ax.axvline(4, color='#444', linewidth=1.5, linestyle='--')

plt.tight_layout()
plt.show()

## Mathematical Formulation

### Inverted Bottleneck

Unlike ResNet (where the bottleneck *reduces* channels: $C \to C/4 \to C$), ConvNeXt uses an **inverted bottleneck** (expand then contract):

$$\text{Inverted Bottleneck: } C \xrightarrow{\text{Linear}} 4C \xrightarrow{\text{GELU}} 4C \xrightarrow{\text{Linear}} C$$

This mirrors MobileNetV2 (Howard et al. 2018) and ViT's MLP blocks.  
The 4× expansion ratio maximises expressiveness per block without increasing output channels.

### Layer Scale (for Large/XLarge variants)

A per-channel learnable scalar $\gamma \in \mathbb{R}^C$ (initialised to $10^{-6}$) multiplies the block output before the residual add:

$$y = x + \gamma \odot f(x)$$

Near-zero initialisation makes each block start as near-identity, improving gradient flow in very deep networks. ConvNeXt V2 replaces this with **GRN** (Global Response Normalisation).

### V1 vs V2: What GRN Adds

**V1 (this notebook)**: no GRN. Layer Scale for large variants. FCMAE pre-training showed V1 blocks collapse features during masked autoencoding (dead filters).

**V2 (Woo et al. 2023)**: adds GRN after GELU:
$$\text{GRN: } x_c \leftarrow \gamma_c \cdot \frac{\|x_c\|_2}{\mathbb{E}[\|x_c\|_2]} \cdot x_c + \beta_c + x_c$$

GRN introduces inter-channel competition — channels with large spatial norms scale up; weak channels are suppressed. This prevents feature collapse during FCMAE self-supervised pre-training.

In [ ]:
"""ConvNeXtV1Block walkthrough: compare with V2."""
import sys
sys.path.insert(0, '../../../src')

import torch
import torch.nn as nn

# Demonstrate the difference: V1 block has no GRN, optionally has Layer Scale
from hale_core.nn.vision.cnn.convnext_v1 import ConvNeXtV1Block
from hale_core.nn.vision.cnn.convnext import ConvNeXtV2Block

v1_tiny_block   = ConvNeXtV1Block(dim=96, layer_scale_init_value=0.0)
v1_large_block  = ConvNeXtV1Block(dim=192, layer_scale_init_value=1e-6)
v2_block        = ConvNeXtV2Block(dim=96)

x = torch.randn(2, 96, 56, 56)  # stride-4 features from stem

print('Block parameter counts:')
def count(m): return sum(p.numel() for p in m.parameters())
print(f'  V1 tiny block  (dim=96,  no layer_scale):  {count(v1_tiny_block):,}')
print(f'  V1 large block (dim=192, layer_scale=1e-6): {count(v1_large_block):,}')
print(f'  V2 block       (dim=96,  with GRN):         {count(v2_block):,}')
print()

# V2 has GRN parameters (gamma + beta = 2 * 4*dim)
v2_extra = count(v2_block) - count(v1_tiny_block)
print(f'V2 extra params (GRN gamma+beta for 4*dim): {v2_extra}')
print(f'  Expected: 2 × 4 × 96 = {2*4*96}')

# Both produce same shape output
y1 = v1_tiny_block(x)
y2 = v2_block(x)
print(f'\nV1 output shape: {y1.shape}')
print(f'V2 output shape: {y2.shape}')

In [ ]:
"""ConvNeXt V1 backbone usage."""
import torch
from hale_core.nn.vision import build_vision_backbone, list_vision_backbones

# List all V1 variants
v1_backbones = sorted([n for n in list_vision_backbones() if 'convnext_v1' in n])
print('ConvNeXt V1 variants:', v1_backbones)

# Compare V1 and V2 base side by side
v1_base = build_vision_backbone('convnext_v1_base', img_size=224)
v2_base = build_vision_backbone('convnext_base',    img_size=224)
v1_base.eval()
v2_base.eval()

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out1 = v1_base(x)
    out2 = v2_base(x)

print(f'\nConvNeXt V1 Base:')
print(f'  pooled_output: {out1.pooled_output.shape}')
print(f'  params: {v1_base.num_parameters()/1e6:.1f}M')
for i, fm in enumerate(out1.feature_maps):
    print(f'  stage {i}: {tuple(fm.shape)}')

print(f'\nConvNeXt V2 Base (for comparison):')
print(f'  pooled_output: {out2.pooled_output.shape}')
print(f'  params: {v2_base.num_parameters()/1e6:.1f}M')
print(f'  (V2 has slightly more params due to GRN gamma/beta)')

# Demonstrate stage freezing for fine-tuning
v1_base.freeze_up_to_stage(1)  # freeze stem + stage 0 + stage 1
trainable = v1_base.num_parameters(trainable_only=True)
total     = v1_base.num_parameters()
print(f'\nAfter freeze_up_to_stage(1):')
print(f'  trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

In [ ]:
"""Compare V1 variants: accuracy vs parameters (published results)."""
import matplotlib.pyplot as plt
import torch
from hale_core.nn.vision import build_vision_backbone

plt.style.use('dark_background')
ACCENT = '#7EB8F7'
GOLD   = '#F7C97E'

variants = [
    ('convnext_v1_tiny',   82.1, 'T'),
    ('convnext_v1_small',  83.1, 'S'),
    ('convnext_v1_base',   83.8, 'B'),
    ('convnext_v1_large',  84.3, 'L'),
]
# Also compare with Swin counterparts (from paper)
swin_variants = [
    ('Swin-T', 81.3, 28),
    ('Swin-S', 83.0, 50),
    ('Swin-B', 83.5, 88),
    ('Swin-L', 84.2, 197),
]

params_v1, accs_v1 = [], []
for name, acc, _ in variants:
    m = build_vision_backbone(name, img_size=224)
    params_v1.append(m.num_parameters() / 1e6)
    accs_v1.append(acc)

swin_params = [sv[2] for sv in swin_variants]
swin_accs   = [sv[1] for sv in swin_variants]
swin_labels = [sv[0] for sv in swin_variants]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(params_v1, accs_v1, s=120, color=ACCENT, zorder=5, label='ConvNeXt V1')
for (name, acc, _), p in zip(variants, params_v1):
    ax.annotate(name.split('_')[-1].upper(), (p, acc), textcoords='offset points',
                xytext=(5, 3), fontsize=8, color=ACCENT)

ax.scatter(swin_params, swin_accs, s=120, color=GOLD, marker='s', zorder=5, label='Swin Transformer')
for label, acc, p in swin_variants:
    ax.annotate(label, (p, acc), textcoords='offset points',
                xytext=(5, -10), fontsize=8, color=GOLD)

ax.set_xlabel('Parameters (M)', color='white')
ax.set_ylabel('ImageNet Top-1 Accuracy (%)', color='white')
ax.set_title('ConvNeXt V1 vs Swin Transformer\n(pure CNN matches attention-based ViT)', color=ACCENT)
ax.legend()
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0d0d1a')
plt.tight_layout()
plt.show()

## References

1. **Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., & Xie, S. (2022).**  
   A ConvNet for the 2020s.  
   *CVPR 2022.* arXiv:2201.03545

2. **Woo, S., Debnath, S., Hu, R., Chen, X., Liu, Z., Kweon, I. S., & Xie, S. (2023).**  
   ConvNeXt V2: Co-designing and Scaling ConvNets with Masked Autoencoders.  
   *CVPR 2023.* arXiv:2301.00808

3. **Liu, Z., Lin, Y., Cao, Y., Hu, H., Wei, Y., Zhang, Z., ... & Guo, B. (2021).**  
   Swin Transformer: Hierarchical Vision Transformer using Shifted Windows.  
   *ICCV 2021.* arXiv:2103.14030

4. **He, K., Zhang, X., Ren, S., & Sun, J. (2016).**  
   Deep Residual Learning for Image Recognition.  
   *CVPR 2016.* arXiv:1512.03385

5. **Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L.-C. (2018).**  
   MobileNetV2: Inverted Residuals and Linear Bottlenecks.  
   *CVPR 2018.* arXiv:1801.04381

6. **Touvron, H., Cord, M., Douze, M., Massa, F., Sablayrolles, A., & Jégou, H. (2021).**  
   Training data-efficient image transformers & distillation through attention (DeiT).  
   *ICML 2021.* arXiv:2012.12877